# 03 - Feature-Level EDA

This notebook analyzes customer-level features from `customer_features.csv` to understand churn patterns and inform modeling.

In [ ]:
# Cell 1: Imports and load customer features
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid')

features_path = Path('../data/processed/customer_features.csv')
customer_df = pd.read_csv(features_path)
print(f'Total customers: {len(customer_df)}')
print(f'Churn rate: {customer_df["Churn"].mean()*100:.2f}%')

In [ ]:
# Cell 2: Target distribution plot
plt.figure(figsize=(6, 4))
customer_df['Churn'].value_counts().plot(kind='bar')
plt.title('Churn Distribution')
plt.xlabel('Churn (0=Active, 1=Churned)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../visualizations/eda/01_churn_distribution.png')
plt.show()

In [ ]:
# Cells 3-6: RFM plots and correlation overview

# 4 RFM plots by churn
rfm_cols = ['Recency', 'Frequency', 'TotalSpent', 'AvgOrderValue']
for col in rfm_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=customer_df, x='Churn', y=col)
    plt.title(f'{col} by Churn')
    plt.tight_layout()
    plt.savefig(f'../visualizations/eda/{col}_by_churn.png')
    plt.show()

# Correlation heatmap for numeric features
plt.figure(figsize=(10, 8))
corr = customer_df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../visualizations/eda/02_correlation_heatmap.png')
plt.show()

# Bar plot of top 10 features most correlated with Churn
churn_corr = corr['Churn'].drop('Churn').sort_values(key=lambda s: s.abs(), ascending=False)
top10 = churn_corr.head(10)
plt.figure(figsize=(8, 4))
top10.plot(kind='bar', color='steelblue')
plt.title('Top 10 Features Correlated with Churn')
plt.ylabel('Correlation with Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/03_top10_corr_with_churn.png')
plt.show()

# Scatter plot: RFM_Score vs TotalSpent colored by Churn
plt.figure(figsize=(6, 4))
sns.scatterplot(data=customer_df, x='RFM_Score', y='TotalSpent', hue='Churn', alpha=0.6)
plt.title('RFM Score vs Total Spent by Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/04_rfm_vs_totalspent_scatter.png')
plt.show()

# Distributions of Recency and RFM_Score by Churn
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.kdeplot(data=customer_df, x='Recency', hue='Churn', common_norm=False)
plt.title('Recency Distribution by Churn')
plt.subplot(1, 2, 2)
sns.kdeplot(data=customer_df, x='RFM_Score', hue='Churn', common_norm=False)
plt.title('RFM Score Distribution by Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/05_recency_rfm_distributions.png')
plt.show()

In [ ]:
# Cells 7-9: Segment and temporal analysis

from scipy import stats

# Segment analysis plots
plt.figure(figsize=(6, 4))
segment_churn = customer_df.groupby('CustomerSegment')['Churn'].mean().sort_values()
segment_churn.plot(kind='bar')
plt.ylabel('Churn Rate')
plt.title('Churn Rate by Customer Segment')
plt.tight_layout()
plt.savefig('../visualizations/eda/06_churn_rate_by_segment.png')
plt.show()

plt.figure(figsize=(6, 4))
segment_sizes = customer_df['CustomerSegment'].value_counts()
segment_sizes.plot(kind='bar')
plt.ylabel('Number of Customers')
plt.title('Customer Segment Sizes')
plt.tight_layout()
plt.savefig('../visualizations/eda/07_segment_sizes.png')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=customer_df, x='Churn', y='RFM_Score')
plt.title('RFM Score by Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/08_rfm_by_churn.png')
plt.show()

# Temporal plots: purchase velocity and recent activity by churn
plt.figure(figsize=(6, 4))
sns.boxplot(data=customer_df, x='Churn', y='PurchaseVelocity')
plt.title('Purchase Velocity by Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/09_purchase_velocity_by_churn.png')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=customer_df, x='Churn', y='Purchases_Last90Days')
plt.title('Purchases in Last 90 Days by Churn')
plt.tight_layout()
plt.savefig('../visualizations/eda/10_recent_activity_by_churn.png')
plt.show()

# Cell 10: T-tests for key numeric features
key_features = ['Recency', 'Frequency', 'TotalSpent', 'RFM_Score', 'ProductDiversityScore', 'PurchaseVelocity']
results = []
for feat in key_features:
    churned = customer_df[customer_df['Churn'] == 1][feat]
    active = customer_df[customer_df['Churn'] == 0][feat]
    t_stat, p_value = stats.ttest_ind(churned, active, equal_var=False)
    results.append({'feature': feat, 't_stat': float(t_stat), 'p_value': float(p_value)})

for r in results:
    print(f"T-test for {r['feature']}: t={r['t_stat']:.4f}, p={r['p_value']:.6f}")
    if r['p_value'] < 0.05:
        print('  -> Significant difference between churned and active customers')
    else:
        print('  -> No significant difference between churned and active customers')